# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhaibachaopls-web/Flyrani_repo/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


#### Lane : Predicting content decay

I am choosing to build a predictive model that flags which currently stable pages are at high risk of losing traffic in the next 30 days. I'm choosing this because this is a binary classificaiton problem which also requires cross vadidation. This path will give me idea oh how to build end to end pilelines, will give me idea on how to pervent data leakage. And I'll also gain experiences with scikitlearn models like XGboost, random forest etc.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df

df['is_decaying'] = np.where(df['trend_pct'] < -20, 1, 0)
class_balance = df['is_decaying'].value_counts(normalize=True) * 100

print(f"Total pages analyzed: {len(df):,}")
print(f"Stable/Growing (0): {class_balance[0]:.2f}%")
print(f"Decaying (1): {class_balance[1]:.2f}%")


Total pages analyzed: 30,000
Stable/Growing (0): 45.81%
Decaying (1): 54.19%


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

This work improves the decision of prioritizing which articles to send to the content/editorial team for a manual refresh before traffic collapses.

Cost of a wrong call:
False positive : Unhealthy pages get flagged as positive and they don't get updated
Flase negative : Healthy pages get flagged as negative and they get updated.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

decaying_pages = df[df['is_decaying'] == 1]
healthy_pages = df[df['is_decaying'] == 0]

total_impressions_at_risk = decaying_pages['impressions_90d'].sum()
avg_impressions_decaying = decaying_pages['impressions_90d'].mean()

print(f"Decaying Pages Count: {len(decaying_pages):,} pages")
print(f"Total 90-day Impressions at Risk: {total_impressions_at_risk:,.0f}")
print(f"Average 90-day Impressions per Decaying Page: {avg_impressions_decaying:,.2f}")




Decaying Pages Count: 16,258 pages
Total 90-day Impressions at Risk: 79,936,075
Average 90-day Impressions per Decaying Page: 4,916.72


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

**High Prevalence of Decay**: Out of 30,000 tracked pages, 54,2% are experiencing a significant negative traffic trend

**Top-Tier Vulnerability**: Over 58% of pages in prime rank positions are actively declining, demonstrating that ranking high today does not guarantee long-term traffic stability.

**Maturity Gap**: Decaying pages have a low median age(~216 days)

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

decay_count = df['is_decaying'].sum()
decay_pct = (decay_count / len(df)) * 100

top_tier_pages = df[df['position_tier'].isin(['page_1', 'striking'])]
top_tier_decay_rate = (top_tier_pages['is_decaying'].sum() / len(top_tier_pages)) * 100


median_age_decaying = decaying_pages['content_age_days'].median()
median_age_stable = healthy_pages['content_age_days'].median()


print(f"1. Decaying Pages Volume: {decay_count:,} ({decay_pct:.1f}% of total dataset)")
print(f"2. Decay Rate in Top Position Tiers (Page 1 & Striking): {top_tier_decay_rate:.1f}%")
print(f"3. Median Content Age — Decaying: {median_age_decaying:.0f} days | Stable: {median_age_stable:.0f} days")




1. Decaying Pages Volume: 16,258 (54.2% of total dataset)
2. Decay Rate in Top Position Tiers (Page 1 & Striking): 58.5%
3. Median Content Age — Decaying: 216 days | Stable: 287 days


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

##### What I can claim
This model provides directional indicators of risk based strictly on observed historical data and measured performance metrics from the provided dataset.

#### What can't be claimed :
This model will never establish causal proof of why traffic dropped, also this model isn't reverse engineering search engine algorithms.



In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


age_decay_corr = df['content_age_days'].corr(df['is_decaying'])

print(f"Observed directional correlation (Content Age vs. Decay Risk): {age_decay_corr:.3f}")
print("Constraint Check 1: This is a measured correlation, NOT causal proof.")
print("Constraint Check 2: Model will output probability scores (0.01 to 0.99) for decision-support, not absolute certainties.")



Observed directional correlation (Content Age vs. Decay Risk): -0.164
Constraint Check 1: This is a measured correlation, NOT causal proof.
Constraint Check 2: Model will output probability scores (0.01 to 0.99) for decision-support, not absolute certainties.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.